## Trims audio from .data annotations into 3s or 5s chunks

In [ ]:
# ============================================================================
#  T R I M   S E G M E N T S   F R O M   A V I A N Z   . D A T A
# ============================================================================
#  Walk a root tree for *.wav.data annotation files, match each to its sibling
#  .wav, and emit time-cropped WAVs under a per-interval output folder.
#
#  Modes:
#    trim_to_annotation           crop each .data segment
#                                  -> {stem}_{n}_trimmed.wav
#    trim_to_interval             ignore annotations, slice whole wav into
#                                  fixed-length windows
#                                  -> {stem}_{n}_trimmed.wav
#    trim_annotation_to_interval  crop each segment, subdivide into windows
#                                  -> {stem}_{n}_{k}_trimmed.wav
#
#  Output folder (per source dir):
#    trim_to_annotation           -> trimmed_to_annotation/
#    trim_to_interval /
#    trim_annotation_to_interval  -> named by the interval itself, e.g. 10sec/
#                                     or 10min/, instead of a generic mode
#                                     folder name. Falls back to
#                                     {mode}_sized/ when tiling is driven by
#                                     max_bytes alone with no interval given.
#  Frequency bounds in each segment row are ignored (time-only crop).
#
#  FILESIZE CONSTRAINT:
#    max_bytes (optional, e.g. 100 * 1024 * 1024 for github's 100MB limit)
#    caps every written clip below a byte ceiling. When set, the effective
#    window length used for interval-based modes is the smaller of the
#    requested interval and a size-safe interval derived from sr/channels/
#    subtype. trim_to_annotation segments that would exceed max_bytes on
#    their own are automatically subdivided using the same size-safe step,
#    even though trim_to_annotation is not normally interval-based.
#
#  INTERVAL INPUT:
#    interval_s (seconds) and interval_min (minutes) are both accepted as
#    duration inputs across trim_one_wav and trim_tree. Exactly one is
#    expected to be set when a duration is required; if both are given,
#    interval_s takes precedence. Resolution happens once via
#    _resolve_interval_seconds so frame-step math and directory-label
#    naming always agree on the same effective duration.
# ============================================================================

import json
import logging
from pathlib import Path

import soundfile as sf




AVIANZ_ROOT = Path("..")                                            # Directory containing main script (AviaNZ.py); relative to avianz/notebooks convention
INPUT_AUDIO_DIR = AVIANZ_ROOT / "test_audio"                        # FALLBACK-TRACK: anchored to AVIANZ_ROOT instead of a separately hardcoded relative path
AUDIO_DIR = str(INPUT_AUDIO_DIR)                                    # Alternate audio source (fewer files for testing purposes) — kept as str for BatchProcessor arg compatibility




# ---- verbose logging (toggle DEBUG_VERBOSE to silence) ---------------------
DEBUG_VERBOSE = True
logging.basicConfig(level=logging.DEBUG if DEBUG_VERBOSE else logging.INFO,
                    format="%(levelname)s %(message)s")
log = logging.getLogger(__name__)

# ---- mode -> output folder name (trim_to_annotation only; interval modes
#      are now named dynamically by duration via _format_interval_label) ----
_MODE_DIRS = {
    "trim_to_annotation": "trimmed_to_annotation",
    "trim_to_interval": "trimmed_to_interval",
    "trim_annotation_to_interval": "trimmed_annotation_to_interval",
}

# ---- subtype -> bytes-per-sample lookup for filesize-safe interval calc ----
# Used only to estimate output clip size before writing (uncompressed WAV,
# so size is deterministic from frame count * channels * bytes_per_sample).
# Unrecognised subtypes fall back to 4 bytes (float32 assumption) rather than
# raising, since this is only a size estimate, not the actual write dtype.
_SUBTYPE_BYTES = {
    "PCM_U8": 1,
    "PCM_S8": 1,
    "PCM_16": 2,
    "PCM_24": 3,
    "PCM_32": 4,
    "FLOAT": 4,
    "DOUBLE": 8,
}


def _bytes_per_frame(subtype, channels):
    """Return estimated bytes-per-frame for a given soundfile subtype and channel count.

    Looks up subtype in _SUBTYPE_BYTES to get bytes-per-sample, then multiplies
    by channels to get bytes-per-frame. Falls back to 4 bytes-per-sample
    (float32 assumption) for unrecognised subtypes, flagged as FALLBACK-TRACK
    since this is an estimate used only for size-safe interval calculation,
    not the actual write path (sf.write still mirrors info.subtype exactly).
    """
    bytes_per_sample = _SUBTYPE_BYTES.get(subtype)
    if bytes_per_sample is None:
        # FALLBACK-TRACK: unrecognised subtype, assume float32 width for
        # size estimation purposes only; actual write still uses info.subtype
        log.warning("  unrecognised subtype %r for size estimate, assuming float32 width", subtype)
        bytes_per_sample = 4
    return bytes_per_sample * channels


def _max_frames_for_size(sr, channels, subtype, max_bytes):
    """Return max frame count per clip so estimated output size stays under max_bytes.

    Divides max_bytes by bytes-per-frame (from _bytes_per_frame) to get a
    frame ceiling. Used to derive a size-safe step equivalent for
    interval-based tiling, and to detect/subdivide oversized annotation
    segments in trim_to_annotation mode.
    """
    bpf = _bytes_per_frame(subtype, channels)
    max_frames = max_bytes // bpf
    return max(1, max_frames)   # FALLBACK-GUARD: never return zero/negative


def _resolve_interval_seconds(interval_s, interval_min):
    """Resolve interval duration in seconds from either interval_s or interval_min input.

    Accepts two mutually optional duration inputs: interval_s (seconds) and
    interval_min (minutes). Exactly one is expected to be non-None when an
    interval-based mode requires a duration; if both are given, interval_s
    takes precedence and interval_min is ignored with a warning, since a
    single unambiguous duration is required downstream for both frame-step
    calculation and directory-label naming.
    """
    if interval_s is not None and interval_min is not None:
        log.warning("  both interval_s and interval_min provided, interval_s takes precedence")
        return interval_s
    if interval_s is not None:
        return interval_s
    if interval_min is not None:
        return interval_min * 60
    return None


def _format_interval_label(interval_s):
    """Return a directory-safe label describing an interval duration in seconds or minutes.

    Whole-minute durations (60s, 120s, etc.) are labeled as Nmin for
    readability; all other durations are labeled as Nsec, using an integer
    form when the value has no fractional part. Returns None when interval_s
    itself is None, signaling that no interval-based label applies (e.g.
    trim_to_annotation mode, or size-only tiling driven by max_bytes alone
    with no interval given).
    """
    if interval_s is None:
        return None
    if interval_s >= 60 and interval_s % 60 == 0:
        minutes = int(interval_s // 60)
        return f"{minutes}min"
    if float(interval_s).is_integer():
        return f"{int(interval_s)}sec"
    return f"{interval_s}sec"


def parse_data_segments(data_path):
    """Return list of (start_s, end_s) segment time bounds from an AviaNZ .data file.

    Reads the JSON array, discards index 0 (header dict: Operator/Reviewer/
    Duration/noiseLevel/noiseTypes), and extracts time bounds from each
    remaining segment row of form [start_s, end_s, low_hz, high_hz, [labels]].
    """
    with open(data_path, "r", encoding="utf-8") as fh:
        payload = json.load(fh)                 # full annotation array

    # index 0 is the metadata header dict; segments are the remaining rows
    segments = []                               # accumulate (start, end) pairs
    for row in payload[1:]:                      # skip header at index 0
        start_s = float(row[0])                 # row[0] = segment start (seconds)
        end_s = float(row[1])                   # row[1] = segment end   (seconds)
        # row[2], row[3] = low_hz, high_hz -> intentionally unused (time-only)
        segments.append((start_s, end_s))
    return segments


def _interval_windows(start_frame, end_frame, step, drop_remainder):
    """Return list of (w_start, w_end) fixed-length frame windows spanning a range.

    Tiles [start_frame, end_frame) into windows of 'step' frames. A trailing
    window shorter than 'step' is kept unless drop_remainder is True.
    """
    windows = []                                # accumulate sub-windows
    cursor = start_frame                        # sliding window start
    while cursor < end_frame:
        w_end = min(cursor + step, end_frame)   # clamp final window to range end
        if (w_end - cursor) < step and drop_remainder:
            break                               # discard trailing short remainder
        windows.append((cursor, w_end))
        cursor += step                          # advance by one window length
    return windows


def trim_one_wav(wav_path, data_path, out_dir, mode="trim_to_annotation",
                 interval_s=None, interval_min=None, drop_remainder=False, max_bytes=None):
    """Crop a single wav into trimmed clips under an explicit output folder.

    mode selects the trimming strategy: 'trim_to_annotation' crops each .data
    segment; 'trim_to_interval' ignores annotations and tiles the whole wav into
    fixed-length windows; 'trim_annotation_to_interval' crops each segment then
    tiles it into fixed-length windows. A duration is required for the two
    interval modes, given as either interval_s (seconds) or interval_min
    (minutes) -- resolved once via _resolve_interval_seconds so the rest of
    this function only deals with interval_s. drop_remainder discards a
    trailing window shorter than the resolved step. Cropping is frame-accurate
    via sample-rate frame conversion.

    out_dir is passed in explicitly (by trim_tree) rather than derived from
    wav_path.parent, so output can be consolidated under a single root instead
    of scattered per-source-folder subdirs.

    max_bytes (optional) caps every written clip below a byte ceiling, e.g.
    100 * 1024 * 1024 for github's 100MB limit. When set:
      - for the two interval modes, the effective step is min(requested
        interval-frame-step, size-safe-frame-step from max_bytes)
      - for trim_to_annotation, any segment whose estimated size exceeds
        max_bytes is automatically subdivided using the size-safe step
        (FALLBACK-TRACK: this borrows interval-tiling logic inside a mode
        that is not normally interval-based, only triggered when a segment
        is oversized)

    Original audio files are never modified -- only read via soundfile.info/read.
    """
    if mode not in _MODE_DIRS:                  # FALLBACK-GUARD: unknown mode
        raise ValueError(f"unknown mode: {mode!r}")

    interval_s = _resolve_interval_seconds(interval_s, interval_min)   # unify seconds/minutes input into a single interval_s before frame-step math

    if mode in ("trim_to_interval", "trim_annotation_to_interval") and not interval_s and not max_bytes:
        # FALLBACK-TRACK: previously required interval_s unconditionally, which
        # blocked size-only tiling (max_bytes as sole constraint). Either
        # interval_s/interval_min or max_bytes alone now satisfies interval-mode
        # requirements -- size_safe_step below becomes the effective step when
        # interval_s is None.
        raise ValueError(f"interval_s, interval_min, or max_bytes required for mode {mode!r}")

    info = sf.info(str(wav_path))               # probe sr / frame count w/o full read
    sr = info.samplerate                        # native sample rate (Hz)
    n_frames_total = info.frames                # total frames for clamping
    channels = info.channels                    # needed for byte-size estimation

    stem = wav_path.stem                        # filename without .wav extension
    out_dir.mkdir(parents=True, exist_ok=True)  # idempotent across repeated runs

    # ---- resolve requested interval step (frames), if given ----------------
    requested_step = int(round(interval_s * sr)) if interval_s else None

    # ---- resolve size-safe step (frames), if max_bytes given ---------------
    size_safe_step = None
    bpf_check = None
    if max_bytes is not None:
        bpf_check = _bytes_per_frame(info.subtype, channels)   # exposed for debug verification below
        size_safe_step = _max_frames_for_size(sr, channels, info.subtype, max_bytes)
        log.debug("  size calc: subtype=%s channels=%d bytes_per_frame=%d",
                  info.subtype, channels, bpf_check)
        log.debug("  size calc: max_bytes=%d size_safe_step=%d frames (%.2fs at sr=%d)",
                  max_bytes, size_safe_step, size_safe_step / sr, sr)
        log.debug("  size calc: estimated clip bytes at size_safe_step = %d (target ceiling %d)",
                  size_safe_step * bpf_check, max_bytes)

    # ---- effective step used for interval-based windowing -------------------
    # smaller of the two constraints wins, so filesize ceiling is respected
    # even when interval_s alone would produce a larger clip. When interval_s
    # is None (size-only tiling), size_safe_step becomes the sole step value.
    if requested_step is not None and size_safe_step is not None:
        step = min(requested_step, size_safe_step)
        log.debug("  step resolution: requested_step=%s size_safe_step=%s -> chosen step=%d",
                  requested_step, size_safe_step, step)
    elif requested_step is not None:
        step = requested_step
    else:
        step = size_safe_step   # size-only tiling path -- interval_s was not provided

    # ---- build the work list of (n, k, w_start, w_end) clip windows --------
    # n indexes the source unit (segment, or whole-file for trim_to_interval);
    # k indexes sub-windows within unit n (None when not subdividing).
    jobs = []                                   # accumulate clip specifications

    if mode == "trim_to_interval":
        # whole-file tiling: single unit n=0, k enumerates windows across file
        for k, (w0, w1) in enumerate(
                _interval_windows(0, n_frames_total, step, drop_remainder)):
            jobs.append((k, None, w0, w1))      # n-slot reused as window index
    else:
        # annotation-driven: iterate segments from the .data file
        segments = parse_data_segments(data_path)
        log.debug("  %d segment(s) in %s", len(segments), data_path.name)
        for n, (start_s, end_s) in enumerate(segments):     # n preserves .data order
            sf0 = max(0, int(round(start_s * sr)))          # segment start frame
            sf1 = min(n_frames_total, int(round(end_s * sr)))   # segment end frame
            if sf1 <= sf0:                      # FALLBACK-GUARD: empty/inverted span
                log.warning("  skip seg %d (empty span %.3f..%.3f s)",
                            n, start_s, end_s)
                continue

            if mode == "trim_to_annotation":
                seg_frames = sf1 - sf0
                if size_safe_step is not None and seg_frames > size_safe_step:
                    # FALLBACK-TRACK: segment exceeds max_bytes on its own,
                    # subdivide using size-safe step even though this mode is
                    # not normally interval-based
                    log.warning("  seg %d exceeds max_bytes as a whole segment, "
                               "subdividing into size-safe windows", n)
                    for k, (w0, w1) in enumerate(
                            _interval_windows(sf0, sf1, size_safe_step, drop_remainder)):
                        jobs.append((n, k, w0, w1))
                else:
                    jobs.append((n, None, sf0, sf1))        # whole segment, no k
            else:  # trim_annotation_to_interval: subdivide this segment
                for k, (w0, w1) in enumerate(
                        _interval_windows(sf0, sf1, step, drop_remainder)):
                    jobs.append((n, k, w0, w1))             # segment n, sub-window k

    # ---- write each clip ----------------------------------------------------
    for n, k, w_start, w_end in jobs:
        block, _ = sf.read(str(wav_path),
                           start=w_start,
                           stop=w_end,
                           dtype="float32")     # float32 keeps full dynamic range

        # 2-index name only when a sub-window index k is present
        if k is None:
            out_name = f"{stem}_{n}_trimmed.wav"
        else:
            out_name = f"{stem}_{n}_{k}_trimmed.wav"
        out_path = out_dir / out_name           # consolidated output root, not source-tree-local
        sf.write(str(out_path), block, sr,
                 subtype=info.subtype)          # mirror source bit-depth/subtype
        log.debug("  wrote %s (%d frames)", out_path.name, w_end - w_start)

        # confirm actual on-disk size against max_bytes ceiling -- this is
        # the only way to verify size_safe_step math produces the intended
        # result rather than assuming it from frame count alone
        if max_bytes is not None:
            actual_size = out_path.stat().st_size
            log.debug("  actual size check: %s = %d bytes (%.2fMB), ceiling=%d bytes (%.1fMB)",
                      out_path.name, actual_size, actual_size / (1024 * 1024),
                      max_bytes, max_bytes / (1024 * 1024))
            if actual_size > max_bytes:
                # FALLBACK-GUARD: size estimate assumptions (subtype lookup,
                # header overhead) can drift slightly from actual written
                # bytes; flag rather than silently exceed the ceiling
                log.warning("  %s exceeded max_bytes ceiling (%d > %d)",
                           out_path.name, actual_size, max_bytes)


def trim_tree(root_dir, mode="trim_to_annotation",
              interval_s=None, interval_min=None, drop_remainder=False,
              output_root=None, max_bytes=None):
    """Recursively trim all wavs that have a sibling .wav.data under root_dir.

    Walks root_dir for *.wav.data files, pairs each with its sibling .wav, and
    trims into a consolidated output_root (default: a 'trimmed_audio' folder
    placed next to root_dir, i.e. a sibling directory rather than nested
    inside it). Relative subfolder structure under root_dir is preserved
    inside output_root/{subfolder_name}/, where subfolder_name is:
      - trimmed_to_annotation, fixed, for trim_to_annotation mode
      - the interval duration itself (e.g. 10sec, 10min) for the two
        interval-based modes, so the folder name reflects what was actually
        trimmed to rather than a generic mode label
      - {mode}_sized as a fallback when tiling is size-only (max_bytes with
        no interval given), since there is no duration to label with

    interval_s / interval_min are resolved once here via
    _resolve_interval_seconds so directory naming and the trim_one_wav call
    both operate on the same effective duration. drop_remainder / max_bytes
    pass through to trim_one_wav unchanged. .data files lacking a sibling wav
    are skipped. Original audio files are never modified.
    """
    root = Path(root_dir)                        # tree root to search
    if output_root is None:                       # FALLBACK-GUARD: default sibling location
        output_root = root.parent / "trimmed_audio"   # placed next to root_dir, not inside it
    else:
        output_root = Path(output_root)

    interval_s = _resolve_interval_seconds(interval_s, interval_min)   # resolved once so naming and trim_one_wav agree on the same effective duration

    data_files = sorted(root.rglob("*.wav.data"))   # every annotation file in tree
    log.info("found %d .wav.data file(s) under %s [mode=%s]",
             len(data_files), root, mode)
    log.info("output routed to %s", output_root)
    if max_bytes is not None:
        log.info("filesize ceiling active: max_bytes=%d (%.1fMB)", max_bytes, max_bytes / (1024 * 1024))

    # ---- determine output subfolder name once (same for every file in tree) ----
    if mode == "trim_to_annotation":
        subfolder_name = _MODE_DIRS[mode]           # fixed name, no interval duration applies
    else:
        interval_label = _format_interval_label(interval_s)
        if interval_label is not None:
            subfolder_name = interval_label         # e.g. "10sec" or "10min"
        else:
            # FALLBACK-TRACK: size-only tiling, no interval duration to label
            # with, fall back to mode folder name with _sized tag
            subfolder_name = f"{_MODE_DIRS[mode]}_sized"
    log.debug("  output subfolder name resolved to: %s", subfolder_name)

    for data_path in data_files:
        # ".wav.data" -> sibling ".wav": strip the trailing ".data" suffix
        wav_path = data_path.with_suffix("")     # drops ".data", leaves ".wav"
        if not wav_path.is_file():               # FALLBACK-GUARD: orphan annotation
            log.warning("no sibling wav for %s -- skipped", data_path.name)
            continue

        # preserve relative subfolder structure from root_dir under output_root
        rel_subdir = wav_path.parent.relative_to(root)   # '.' when wav sits directly in root
        out_dir = output_root / subfolder_name / rel_subdir

        log.info("trimming %s", wav_path.relative_to(root))
        trim_one_wav(wav_path, data_path, out_dir, mode=mode,
                     interval_s=interval_s, drop_remainder=drop_remainder,
                     max_bytes=max_bytes)


# ============================================================================
#  U S A G I
# ============================================================================
# from trim_segments import trim_tree
#
# # crop each annotation segment only (ignore interval slicing entirely):
# # -> output/trimmed_to_annotation/
# trim_tree("test_audio", mode="trim_to_annotation")
#
# # tile whole wav into 3 s windows (annotations ignored):
# # -> output/3sec/
# trim_tree("test_audio", mode="trim_to_interval", interval_s=3)
#
# # tile whole wav into 10 s windows (annotations ignored):
# # -> output/10sec/
# trim_tree("test_audio", mode="trim_to_interval", interval_s=10)
#
# # tile whole wav into 10 minute windows using interval_min instead of
# # interval_s -- both accepted, interval_min converts to seconds internally:
# # -> output/10min/
# trim_tree("test_audio", mode="trim_to_interval", interval_min=10)
#
# # crop segments, then subdivide each into 3 s windows, drop short tails:
# # -> output/3sec/
# trim_tree("test_audio", mode="trim_annotation_to_interval",
#           interval_s=3, drop_remainder=True)
#
# # SIZE-ONLY TILING: no interval_s/interval_min given -- size_safe_step
# # alone drives window length, clips land close to the max_bytes ceiling:
# # -> output/trimmed_to_interval_sized/
# trim_tree("test_audio", mode="trim_to_interval",
#           max_bytes=100 * 1024 * 1024)
#
# # COMBINING an interval with max_bytes: the SMALLER of the two constraints
# # always wins. Passing interval_s=10 alongside max_bytes=100MB will still
# # produce ~10s clips, since 10s is far below the 100MB ceiling in almost
# # all cases -- only omit interval_s/interval_min when the goal is clips
# # sized close to max_bytes itself.
# trim_tree("test_audio", mode="trim_to_interval", interval_s=10,
#           max_bytes=100 * 1024 * 1024)   # -> still ~10s clips, folder "10sec"
#
# # all modes above write into a sibling "trimmed_audio" folder next to
# # "test_audio" by default (output_root override available if needed)


# Trim audio files to fixed-length windows and save under a duration-named
# subfolder inside a "trimmed_audio" dir placed next to AUDIO_DIR (not
# nested inside it).

# trim_tree(AUDIO_DIR, mode="trim_to_interval", interval_s=10 * 60)

trim_tree(AUDIO_DIR, mode="trim_to_interval", interval_min=10)   # example call -- swap interval_s, interval_min, mode, or max_bytes as needed



INFO found 4 .wav.data file(s) under ..\test_audio [mode=trim_to_interval]
INFO output routed to ..\trimmed_audio
DEBUG   output subfolder name resolved to: 10min
INFO trimming andre_ams3_20230808_000500.WAV
DEBUG   wrote andre_ams3_20230808_000500_0_trimmed.wav (2640000 frames)
INFO trimming andre_ams3_20230808_001100.WAV
DEBUG   wrote andre_ams3_20230808_001100_0_trimmed.wav (2640000 frames)
INFO trimming andre_c2_20230809_233017.wav
DEBUG   wrote andre_c2_20230809_233017_0_trimmed.wav (28800000 frames)
DEBUG   wrote andre_c2_20230809_233017_1_trimmed.wav (28800000 frames)
DEBUG   wrote andre_c2_20230809_233017_2_trimmed.wav (6121 frames)
INFO trimming martin_c6_20230812_040002.wav
DEBUG   wrote martin_c6_20230812_040002_0_trimmed.wav (28800000 frames)
DEBUG   wrote martin_c6_20230812_040002_1_trimmed.wav (28800000 frames)
DEBUG   wrote martin_c6_20230812_040002_2_trimmed.wav (6121 frames)
